he DPO Derivation

The goal of this lesson is just to understand how DPO gets rid of the reward model.

We already know:

\(x\) = prompt
\(y_w\) = chosen answer
\(y_l\) = rejected answer
\(\pi_\theta\) = our LLM
\(\pi_{\rm ref}\) = frozen reference LLM
\(r(x,y)\) = hypothetical reward/quality score
1. Start from the old RLHF idea

Imagine we somehow know the reward:

$$ r(x,y) $$

We want our LLM to generate high-reward answers.

The naive objective would be:

$$ \max_\pi E[r(x,y)] $$

Meaning:

Make the model generate answers with higher reward.

But there's a problem.

If we only maximize reward, the model could go crazy.

It might move very far away from the model we started with.

So RLHF adds a KL penalty:

$$ \boxed{ \max_\pi E[r(x,y)] - \beta KL(\pi||\pi_{\rm ref}) } $$

Don't worry about the KL details yet.

Conceptually this says:

Get better according to the reward, but don't move too far away from the reference model.

2. What does the reference model actually mean?

Suppose our SFT model says:

"The cat sat on the mat."

with some probability.

That's our starting/reference behavior:

$$ \pi_{\rm ref} $$

Then our trainable model:

$$ \pi_\theta $$

is allowed to change.

So:

SFT model
   │
   └── frozen copy → πref

SFT model
   │
   └── train → πθ

The reference is basically our anchor.

3. Here's the key mathematical result

If you optimize:

$$ E[r]-\beta KL(\pi||\pi_{\rm ref}) $$

the optimal policy has the form:

$$ \boxed{ \pi^*(y|x) \propto \pi_{\rm ref}(y|x) e^{r(x,y)/\beta} } $$

This looks scary, but read it literally.

The new probability is roughly:

$$ \text{new probability} = \text{old probability} \times \text{reward boost} $$

So:

If reward is high
$$ e^{r/\beta} $$

is large.

Therefore the new policy gives that answer more probability.

If reward is low

the multiplier is smaller.

Therefore the new policy gives it less probability.

4. "Wait, where did this equation come from?"

Good question.

We don't need to do a 3-page calculus proof.

The optimization is basically asking:

For every possible answer, how should I redistribute probability so that reward goes up while staying close to the reference?

The mathematical solution to that constrained optimization gives:

$$ \pi^*(y|x) = \frac{1}{Z(x)} \pi_{\rm ref}(y|x)e^{r(x,y)/\beta} $$

where \(Z(x)\) is just a normalization constant.

You can think of \(Z\) as:

"Whatever number is necessary so all the probabilities add up to 1."

5. Now do the important rearrangement

We have:

$$ \pi^*(y|x) = \frac{1}{Z(x)} \pi_{\rm ref}(y|x)e^{r(x,y)/\beta} $$

Take log:

$$ \log\pi^*(y|x) = \log\pi_{\rm ref}(y|x) + \frac{r(x,y)}{\beta} - \log Z(x) $$

Rearrange:

$$ \frac{r(x,y)}{\beta} = \log\pi^*(y|x) - \log\pi_{\rm ref}(y|x) + \log Z(x) $$

Therefore:

$$ \boxed{ r(x,y) = \beta \left[ \log\frac{\pi^*(y|x)} {\pi_{\rm ref}(y|x)} + \log Z(x) \right] } $$

And here's the DPO magic.

6. We don't actually care about the absolute reward

Remember Bradley-Terry?

It needed:

$$ r(x,y_w)-r(x,y_l) $$

So let's calculate that.

For chosen:

$$ r_w = \beta \left[ \log\frac{\pi^*(y_w|x)} {\pi_{\rm ref}(y_w|x)} + \log Z(x) \right] $$

For rejected:

$$ r_l = \beta \left[ \log\frac{\pi^*(y_l|x)} {\pi_{\rm ref}(y_l|x)} + \log Z(x) \right] $$

Subtract:

$$ r_w-r_l $$

The \(Z(x)\) terms cancel:

$$ \boxed{ r_w-r_l = \beta \left[ \log\frac{\pi^*(y_w|x)} {\pi_{\rm ref}(y_w|x)} - \log\frac{\pi^*(y_l|x)} {\pi_{\rm ref}(y_l|x)} \right] } $$

BOOM.

That's the entire trick.

We started with a hypothetical reward.

But the reward difference can be calculated entirely from policy probabilities and reference probabilities.

So we don't need to build a reward model.

7. Now plug this into Bradley-Terry

Bradley-Terry said:

$$ P(y_w\succ y_l|x) = \sigma(r_w-r_l) $$

We just found:

$$ r_w-r_l = \beta \left[ \log\frac{\pi^*(y_w|x)} {\pi_{\rm ref}(y_w|x)} - \log\frac{\pi^*(y_l|x)} {\pi_{\rm ref}(y_l|x)} \right] $$

Therefore:

$$ P(y_w\succ y_l|x) = \sigma \left( \beta \left[ \log\frac{\pi^*(y_w|x)} {\pi_{\rm ref}(y_w|x)} - \log\frac{\pi^*(y_l|x)} {\pi_{\rm ref}(y_l|x)} \right] \right) $$

Now comes one final thing.

We don't know \(\pi^*\).

We're trying to train \(\pi_\theta\) to become better.

So replace:

$$ \pi^* $$

with:

$$ \pi_\theta $$
8. And we get DPO

Our model predicts the probability that the chosen answer should win:

$$ P_{\theta}(y_w\succ y_l|x) $$

Then we maximize that probability.

Equivalent loss:

$$ \boxed{ L_{\rm DPO} = -\log\sigma \left( \beta \left[ \log\frac{\pi_\theta(y_w|x)} {\pi_{\rm ref}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)} {\pi_{\rm ref}(y_l|x)} \right] \right) } $$

That's DPO.

9. Let's make the formula human-readable

Define:

$$ \text{chosen ratio} = \frac{\pi_\theta(y_w|x)} {\pi_{\rm ref}(y_w|x)} $$

and:

$$ \text{rejected ratio} = \frac{\pi_\theta(y_l|x)} {\pi_{\rm ref}(y_l|x)} $$

DPO essentially asks:

Has our new model increased the chosen answer's probability relative to the reference more than it increased the rejected answer's probability?

If yes → good.

If no → bad.

That's the intuition you should keep.

And now the thing you were confused about earlier

There are three probabilities/scores floating around:

Human data

No probability.

Just:

$$ y_w\succ y_l $$
Bradley-Terry model

Predicts:

$$ P(y_w\succ y_l) $$

based on a hypothetical reward difference.

LLM

Produces:

$$ \pi_\theta(y|x) $$

which is the LLM's probability of generating the response.

DPO mathematically connects the third to the second, allowing us to eliminate the explicit reward model.

12. The complete trick in one line

This is the line I want you to remember:

$$ \boxed{ \text{KL-constrained RL} \Rightarrow r(x,y) \leftrightarrow \log\frac{\pi_\theta(y|x)} {\pi_{\rm ref}(y|x)} } $$

Then:

$$ \boxed{ \text{Bradley-Terry} + \text{that substitution} = \text{DPO} } $$

So DPO isn't randomly inventing a new loss.

It's basically:

"We know what the optimal KL-regularized RL policy looks like, so let's express the reward through the policy instead of separately learning a reward model."

First, tiny notation correction:

\(\pi_\theta\) = current trainable LLM
\(\pi_{\text{ref}}\) = frozen reference LLM
\(r\) = hypothetical reward, not a model we're training
So what is \(\pi_{\text{ref}}\)?

It's basically a frozen copy of the model BEFORE DPO training.

Say we start with an SFT model:

                SFT MODEL
                   │
          ┌────────┴────────┐
          ↓                 ↓
    trainable copy      frozen copy
      πθ                  πref

They initially have identical weights.

Example

Prompt:

"Explain gravity."

Chosen answer:

"Gravity is the force that attracts objects with mass..."

Rejected answer:

"Gravity is when things fall down."

Before DPO

The SFT/reference model might assign:

$$ \pi_{\text{ref}}(y_w|x)=0.10 $$ $$ \pi_{\text{ref}}(y_l|x)=0.08 $$

These are:

How much probability the original SFT model gives to each complete response.

Then DPO trains \(\pi_\theta\).

After some training, suppose:

$$ \pi_\theta(y_w|x)=0.20 $$ $$ \pi_\theta(y_l|x)=0.04 $$

Now DPO sees:

Chosen
$$ \frac{\pi_\theta(y_w|x)} {\pi_{\text{ref}}(y_w|x)} = \frac{0.20}{0.10} =2 $$

The new model made the chosen answer 2× more likely relative to the original model.

Rejected
$$ \frac{\pi_\theta(y_l|x)} {\pi_{\text{ref}}(y_l|x)} = \frac{0.04}{0.08} =0.5 $$

The new model made the rejected answer half as likely relative to the original model.

That's what DPO wants.